# Step 16 — CIFAR-10 follow-up: coordinate robustness, the sign of h, and a noise-controlled fidelity estimate

1. **Two classifiers (seeds 0, 1), two censuses.** Per-pair correlation of $h$ and $\sigma_{\rm bif}$ between coordinates: is the census a property of the data pair or of one network?
2. **Sign of $h$ vs class-variance ordering** across 45 pairs.
3. **Noise-bias test for $\sigma_{\rm bif}$** (CPU): add Gaussian noise at the measured split-half level to the exact curve, re-bisect, and see whether the transition shifts upward.
4. **Improved fidelity estimator**: exact-JVP Hutchinson with 32 probes for the divergence term, winsorised bin means, and $\sigma_{\rm bif}$ from each split half separately, on four pairs.

Self-contained (retrains the classifiers, ~13 min each, and saves them). GPU on, Internet on. ~3.5 h total; parts 1–3 finish in ~2.5 h.

In [ ]:
import os; os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import sys, subprocess, pickle, numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math, time
import matplotlib.pyplot as plt
from torchvision import datasets
from scipy.signal import fftconvolve
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print(dev); rng = np.random.default_rng(0); torch.manual_seed(0); plt.rcParams['figure.dpi'] = 130

GRID = np.linspace(-4.0, 4.0, 4001); DG = GRID[1]-GRID[0]
def ref_score(m):
    """Marginal score of the standardized order parameter m on GRID from a Silverman-bandwidth KDE (m is unbounded and smooth)."""
    m = np.clip(m, -8, 8); bw = max(1.5*1.06*m.std()*len(m)**(-1/5), 1e-3)
    h, _ = np.histogram(m, bins=np.append(GRID-DG/2, GRID[-1]+DG/2), density=True); k = np.exp(-GRID**2/(2*bw**2)); k /= k.sum(); p = np.maximum(fftconvolve(h, k, 'same'), 1e-300)
    return np.gradient(np.log(p), GRID), p
def _merge(pts, merge=0.02):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary_points(sm, p=None, lo=-3.5, hi=3.5, prominence=0.05):
    """Zero crossings of a marginal score sm on GRID; adjacent max/min pairs whose log-density differs by < prominence nats are dropped."""
    keep = (GRID > lo) & (GRID < hi)
    if p is not None: keep &= p > 1e-3*p.max()
    idx = np.where((np.sign(sm[:-1]) != np.sign(sm[1:])) & keep[:-1] & keep[1:])[0]
    pts = _merge([(float(GRID[i]-sm[i]*DG/(sm[i+1]-sm[i])) if sm[i+1] != sm[i] else float(GRID[i]), 'max' if sm[i] > 0 else 'min') for i in idx], merge=0.03)
    logp = np.concatenate([[0.0], np.cumsum(0.5*(sm[1:]+sm[:-1])*DG)]); lp = lambda m: float(np.interp(m, GRID, logp))
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if pts[a][1] != pts[a+1][1] and abs(lp(pts[a][0]) - lp(pts[a+1][0])) < prominence: del pts[a:a+2]; ch = True; break
    return pts
def n_max(pts): return sum(k == 'max' for _, k in pts)
def zero_crossings(cen, mean, cnt, min_count=40):
    okk = cnt >= min_count; c, f = cen[okk], mean[okk]; out = []
    for i in range(len(c)-1):
        if np.sign(f[i]) != np.sign(f[i+1]) and f[i] != 0: out.append((float(c[i]-f[i]*(c[i+1]-c[i])/(f[i+1]-f[i])), 'max' if f[i] > 0 else 'min'))
    return _merge(out, merge=0.05)
def cusp_fit(m, f, w, window=1.0, b_min=0.02, s_max=0.6):
    ok = (np.abs(m) < window) & (w > 0)
    if ok.sum() < 8: return dict(h=np.nan, a=np.nan, b=np.nan, clamped=True)
    c3, c2, c1, c0 = np.polyfit(m[ok], f[ok], 3, w=np.sqrt(w[ok])); b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    return dict(h=c0 + c1*s + c2*s**2 + c3*s**3, a=c1 + 3*c3*s**2 + 2*c2*s, b=b, clamped=(-c3 <= b_min+1e-12))
def bisect_sigma(get_pts, sigmas, iters=8):
    """Largest sigma at which the exact marginal of the order parameter is bimodal (scan downward in sigma)."""
    ss = sorted(sigmas, reverse=True); hi = None
    for s in ss:
        pts = get_pts(s)
        if n_max(pts) >= 2 and hi is not None: lo = s; break
        if n_max(pts) == 1: hi = s
    else: return None
    for _ in range(iters):
        mid = np.sqrt(lo*hi)
        if n_max(get_pts(mid)) >= 2: lo = mid
        else: hi = mid
    pm = [p for p, k in get_pts(hi) if k == 'max']; prev = pm[0] if pm else 0.0; pl = get_pts(lo)
    mx = sorted(p for p, k in pl if k == 'max'); mn = [p for p, k in pl if k == 'min']
    return float(lo), float(abs((mn[0] if mn else 0.5*(mx[0]+mx[-1])) - prev))

def mean_force_nonlinear(phi_fn, score_fn, x, sigma, n_hutch=4, eps=1e-2):
    """Per-sample estimate of  grad(phi).s/|grad phi|^2 + div(grad phi/|grad phi|^2)  at x (batch), with score s = score_fn(x, sigma).
    Divergence via Hutchinson with finite-difference JVPs of F(x) = grad phi/|grad phi|^2."""
    x = x.detach().requires_grad_(True)
    def Fvec(xx):
        xx = xx.detach().requires_grad_(True); ph = phi_fn(xx, sigma); g = torch.autograd.grad(ph.sum(), xx)[0]; return g/(g.pow(2).flatten(1).sum(1).view(-1, *[1]*(x.dim()-1)) + 1e-12)
    g = Fvec(x); s = score_fn(x, sigma).detach(); term1 = (g*s).flatten(1).sum(1)
    div = torch.zeros(len(x), device=x.device)
    for _ in range(n_hutch):
        v = torch.randn_like(x); v = v/v.flatten(1).norm(dim=1).view(-1, *[1]*(x.dim()-1))*math.sqrt(x[0].numel())
        Jv = (Fvec(x+eps*v) - Fvec(x-eps*v))/(2*eps); div += (Jv*v).flatten(1).sum(1)
    return (term1 + div/n_hutch).detach()

In [ ]:
if not os.path.exists('/kaggle/working/edm'): subprocess.run(['git', 'clone', '-q', 'https://github.com/NVlabs/edm.git', '/kaggle/working/edm'], check=True)
sys.path.insert(0, '/kaggle/working/edm'); import dnnlib
ds = datasets.CIFAR10('/kaggle/working/cifar', train=True, download=True)
X = torch.from_numpy((np.asarray(ds.data).astype(np.float32)/127.5 - 1.0).transpose(0, 3, 1, 2)); Y = np.asarray(ds.targets)
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
with dnnlib.util.open_url('https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/edm-cifar10-32x32-uncond-vp.pkl') as f: net = pickle.load(f)['ema'].to(dev).eval()
def score_net(x, sigma):
    with torch.no_grad(): D = net(x, torch.full((len(x),), float(sigma), device=dev))
    return (D - x)/sigma**2

class FiLMBlock(nn.Module):
    def __init__(self, cin, cout, emb, stride=1):
        super().__init__(); self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout); self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False); self.b2 = nn.BatchNorm2d(cout)
        self.film = nn.Linear(emb, 2*cout); self.sc = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout)) if (stride != 1 or cin != cout) else nn.Identity()
    def forward(self, x, e):
        g, b = self.film(e).chunk(2, 1); h = F.silu(self.b1(self.c1(x))); h = self.b2(self.c2(h))*(1+g[:, :, None, None]) + b[:, :, None, None]; return F.silu(h + self.sc(x))
class NoisyResNet(nn.Module):
    """10-class noise-conditional classifier (small ResNet with FiLM conditioning on log sigma)."""
    def __init__(self, ch=64, emb=128):
        super().__init__(); self.emb = nn.Sequential(nn.Linear(32, emb), nn.SiLU(), nn.Linear(emb, emb)); self.inp = nn.Conv2d(3, ch, 3, 1, 1)
        self.blocks = nn.ModuleList([FiLMBlock(ch, ch, emb), FiLMBlock(ch, 2*ch, emb, 2), FiLMBlock(2*ch, 2*ch, emb), FiLMBlock(2*ch, 4*ch, emb, 2), FiLMBlock(4*ch, 4*ch, emb)]); self.fc = nn.Linear(4*ch, 10)
    def forward(self, x, sigma):
        f = torch.exp(torch.arange(16, device=x.device)*-(math.log(100)/15)); t = 0.25*torch.log(sigma)[:, None]*f; e = self.emb(torch.cat([torch.sin(t), torch.cos(t)], 1))
        h = self.inp(x)
        for b in self.blocks: h = b(h, e)
        return self.fc(h.mean((2, 3)))
def train_clf10(seed=0, epochs=25, bs=256, s_lo=0.05, s_hi=10.0):
    torch.manual_seed(seed)
    Xd = X.to(dev); Yd = torch.from_numpy(Y).long().to(dev); clf = NoisyResNet().to(dev); opt = torch.optim.AdamW(clf.parameters(), 2e-3, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=epochs*(len(Xd)//bs)); t0 = time.time()
    for ep in range(epochs):
        clf.train(); perm = torch.randperm(len(Xd), device=dev)
        for i in range(0, len(Xd)-bs+1, bs):
            j = perm[i:i+bs]; x0 = Xd[j]
            if torch.rand(1).item() < 0.5: x0 = x0.flip(3)                       # horizontal flip
            sigma = torch.exp(torch.empty(bs, device=dev).uniform_(math.log(s_lo), math.log(s_hi))); xt = x0 + sigma.view(-1, 1, 1, 1)*torch.randn_like(x0)
            loss = F.cross_entropy(clf(xt, sigma), Yd[j]); opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if ep % 5 == 4 or ep == epochs-1:
            clf.eval()
            with torch.no_grad():
                acc = {}
                for sg in [0.05, 0.5, 1.0, 2.0, 4.0]:
                    xs = Xd[:5000] + sg*torch.randn_like(Xd[:5000]); pred = torch.cat([clf(xb, torch.full((len(xb),), sg, device=dev)).argmax(1) for xb in xs.split(500)]); acc[sg] = float((pred == Yd[:5000]).float().mean())
            print(f"  epoch {ep+1}: train-set accuracy at sigma 0.05/0.5/1/2/4 = " + '/'.join(f"{acc[k]:.2f}" for k in acc) + f"  ({time.time()-t0:.0f}s)")
    clf.eval(); torch.save(clf.state_dict(), f'/kaggle/working/noisy_resnet_seed{seed}.pt'); return clf
CLF10 = None
@torch.no_grad()
def clf_batched(x, sigma, bs=1000):
    return torch.cat([CLF10(xb, torch.full((len(xb),), float(sigma), device=dev)) for xb in x.split(bs)])
def pair_logit(pair):
    """Callable (x, sigma) -> log-odds of class a vs b from the 10-class classifier. Batched internally when no grad is needed."""
    def f(x, sg):
        if len(x) <= 1000: l = CLF10(x, torch.full((len(x),), float(sg), device=dev))     # direct (differentiable) for small batches
        else: l = clf_batched(x, sg)                                                          # batched, no-grad, for reference sampling
        return l[:, pair[0]] - l[:, pair[1]]
    return f

SIGMAS = np.geomspace(0.1, 30.0, 30)
def standardizer(clf, Xp, Yp, sigma, n=20000):
    """(mid, half, separability) of the pair log-odds at this sigma from class-conditional means."""
    with torch.no_grad():
        j = torch.randint(len(Xp), (n,), device=dev); xt = Xp[j] + sigma*torch.randn_like(Xp[j]); z = clf(xt, sigma); y = Yp[j] > 0.5; del xt
        mu_a, mu_b = z[y].mean(), z[~y].mean(); half = 0.5*(mu_a - mu_b); sep = float(half.abs()/(0.5*(z[y].std()+z[~y].std())))
    return float(0.5*(mu_a+mu_b)), float(half), sep
def exact_census(clf, Xp, Yp, sigmas=SIGMAS, n=20000, min_sep=0.3):
    """Exact marginal score of the standardized logit over forward-noised pair data; stationary points; bisected transition; cusp fit."""
    cache = {}
    def ref_at(sigma):
        key = round(float(sigma), 5)
        if key not in cache:
            mid, half, sep = standardizer(clf, Xp, Yp, sigma)
            if sep < min_sep: cache[key] = (None, None, None, sep); return cache[key]
            with torch.no_grad():
                m = []
                for _ in range(n//5000):
                    j = torch.randint(len(Xp), (5000,), device=dev); xt = Xp[j] + sigma*torch.randn_like(Xp[j]); m.append(((clf(xt, sigma) - mid)/half).cpu().numpy()); del xt
            m = np.concatenate(m); sm, p = ref_score(m); cache[key] = (sm, p, m, sep)
        return cache[key]
    def pts_at(sigma):
        sm, p, m, sep = ref_at(sigma); return ([(0.0, 'max')] if sm is None else stationary_points(sm, p)), m
    bj = bisect_sigma(lambda s: pts_at(s)[0], sigmas, iters=6)
    if bj is None: return None
    sm, p, m, sep = ref_at(bj[0]); f = cusp_fit(GRID, sm, p)
    return dict(sigma_bif=bj[0], jump=bj[1], sep_at_bif=sep, **f, pts_at=pts_at, ref_at=ref_at, std=lambda sg: standardizer(clf, Xp, Yp, sg))

from itertools import combinations
def pair_data(pair):
    idx = np.where(np.isin(Y, pair))[0]; return X[idx].to(dev), torch.from_numpy((Y[idx] == pair[0]).astype(np.float32)).to(dev)
def run_census(tag):
    out = {}
    for pair in combinations(range(10), 2):
        Xp, Yp = pair_data(pair); clf = pair_logit(pair); r = exact_census(clf, Xp, Yp)
        out[pair] = None if r is None else dict(sigma_bif=r['sigma_bif'], jump=r['jump'], h=r['h'], a=r['a'], b=r['b'], clamped=r['clamped'], sep=r['sep_at_bif'])
    print(f"census [{tag}]: {sum(v is not None for v in out.values())}/45 bimodal"); return out


## 1. Two coordinates, two censuses

In [ ]:
import glob
CLFS = {}
for seed in [0, 1]:
    found = glob.glob(f'/kaggle/input/**/noisy_resnet_seed{seed}.pt', recursive=True)
    if found:
        CLFS[seed] = NoisyResNet().to(dev); CLFS[seed].load_state_dict(torch.load(found[0], map_location=dev)); CLFS[seed].eval(); print(f'loaded classifier seed {seed} from {found[0]}')
    else:
        CLFS[seed] = train_clf10(seed=seed)
CEN = {}
for seed in [0, 1]:
    CLF10 = CLFS[seed]; CEN[seed] = run_census(f'seed {seed}')
both = [p for p in CEN[0] if CEN[0][p] and CEN[1][p]]
h0 = np.array([CEN[0][p]['h'] for p in both]); h1 = np.array([CEN[1][p]['h'] for p in both]); s0 = np.array([CEN[0][p]['sigma_bif'] for p in both]); s1 = np.array([CEN[1][p]['sigma_bif'] for p in both])
print(f"pairs bimodal under both coordinates: {len(both)}/45")
print(f"corr(h seed0, h seed1) = {np.corrcoef(h0, h1)[0,1]:.3f}; sign agreement {np.mean(np.sign(h0) == np.sign(h1)):.2f}; median |h0-h1| = {np.median(np.abs(h0-h1)):.3f} (median |h| = {np.median(np.abs(h0)):.3f})")
print(f"corr(log sigma_bif seed0, seed1) = {np.corrcoef(np.log(s0), np.log(s1))[0,1]:.3f}; median ratio {np.median(s1/s0):.2f}")
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6)); ax[0].scatter(h0, h1, s=14); ax[0].axline((0, 0), slope=1, color='k', lw=0.6); ax[0].set_xlabel('h (classifier seed 0)'); ax[0].set_ylabel('h (classifier seed 1)')
ax[1].scatter(s0, s1, s=14); ax[1].set_xscale('log'); ax[1].set_yscale('log')
lim = [min(s0.min(), s1.min())*0.8, max(s0.max(), s1.max())*1.2]; ax[1].plot(lim, lim, color='k', lw=0.6)
ax[1].set_xlabel('sigma_bif (seed 0)'); ax[1].set_ylabel('sigma_bif (seed 1)'); plt.tight_layout(); plt.show()
VEH = {0, 1, 8, 9}
kind = ['vehicle/vehicle' if (p[0] in VEH and p[1] in VEH) else 'animal/animal' if (p[0] not in VEH and p[1] not in VEH) else 'vehicle/animal' for p in both]
for k in ['vehicle/animal', 'vehicle/vehicle', 'animal/animal']:
    v = [s0[i] for i in range(len(both)) if kind[i] == k]; print(f"  sigma_bif for {k:>16}: median {np.median(v):.2f} (n={len(v)}, range {min(v):.2f}-{max(v):.2f})")

## 2. Sign of h vs class-variance ordering

In [ ]:
var = {c: float(X[Y == c].var(0).mean()) for c in range(10)}
print('class pixel variance:', {CLASSES[c]: round(v, 4) for c, v in var.items()})
# convention: h>0 means the '+' state (class a, the FIRST of the pair) persists through the transition and class b is born at the fold
agree = []; rows = []
for p in both:
    hh = 0.5*(CEN[0][p]['h'] + CEN[1][p]['h']); higher_var_is_a = var[p[0]] > var[p[1]]
    persists_a = hh > 0; agree.append(persists_a == higher_var_is_a); rows.append((p, hh, var[p[0]]-var[p[1]]))
agree = np.array(agree); print(f"'higher-variance class persists' holds for {agree.sum()}/{len(agree)} pairs")
hv = np.array([r[1] for r in rows]); dv = np.array([r[2] for r in rows]); print(f"corr(h, var_a - var_b) = {np.corrcoef(hv, dv)[0,1]:.3f}")
plt.figure(figsize=(4.5, 3.6)); plt.scatter(dv, hv, s=14); plt.axhline(0, color='k', lw=0.5); plt.axvline(0, color='k', lw=0.5); plt.xlabel('var(class a) - var(class b)'); plt.ylabel('h (mean of two coordinates)'); plt.tight_layout(); plt.show()

## 3. Does estimator noise bias sigma_bif upward? (CPU test on the exact curve)

In [ ]:
CLF10 = CLFS[0]
def noisy_bisect(pair, rel_noise, trials=8, seed=0):
    Xp, Yp = pair_data(pair); clf = pair_logit(pair); r = exact_census(clf, Xp, Yp); rs = np.random.default_rng(seed); out = []
    for t in range(trials):
        def pts(sg):
            sm, p, m, sep = r['ref_at'](sg)
            if sm is None: return [(0.0, 'max')]
            scale = np.sqrt(np.mean(sm[(GRID > -1.5) & (GRID < 1.5)]**2)); noise = rs.standard_normal(len(sm))*rel_noise*scale
            k = np.exp(-0.5*(np.arange(-15, 16)/6.0)**2); k /= k.sum(); noise = np.convolve(noise, k, 'same')*np.sqrt(len(k)/np.sum(k**2))/np.sqrt(len(k))  # correlated like binned+smoothed noise
            return stationary_points(sm + noise, p)
        bj = bisect_sigma(pts, SIGMAS, iters=6); out.append(bj[0] if bj else np.nan)
    return r['sigma_bif'], np.array(out)
for pair in [(2, 3), (3, 9), (5, 7), (3, 7)]:
    ex, noisy = noisy_bisect(pair, 0.32)
    print(f"{CLASSES[pair[0]]}/{CLASSES[pair[1]]}: exact sigma_bif {ex:.3f}; with 32% correlated noise: median {np.nanmedian(noisy):.3f}, range {np.nanmin(noisy):.3f}-{np.nanmax(noisy):.3f}  (ratio {np.nanmedian(noisy)/ex:.2f})")
print("If the ratio is ~1, the 30-50% upward shift of the network's sigma_bif is not a noise artifact.")

## 4. Fidelity with a lower-variance estimator (exact JVP, 32 probes, winsorised bins)

In [ ]:
# ---- Step 16, part 4 (replacement): fidelity with a lower-variance, memory-safe estimator ----
import gc; torch.cuda.empty_cache(); gc.collect()
CLF10 = CLFS[0]

def mean_force_batched(phi_fn, score_fn, x, sigma, n_hutch=16):
    """grad(phi).s/|grad phi|^2 + div(grad phi/|grad phi|^2) per sample, with the divergence from exact JVPs of the batched
    field F(x) = grad phi/|grad phi|^2 (double backward), Rademacher probes. Memory ~ a few passes of the batch."""
    x = x.detach()
    def F_batched(xx):
        xx = xx.requires_grad_(True); ph = phi_fn(xx, sigma); g = torch.autograd.grad(ph.sum(), xx, create_graph=True)[0]
        return g/(g.pow(2).flatten(1).sum(1).view(-1, 1, 1, 1) + 1e-12)
    with torch.no_grad(): s = score_fn(x, sigma)
    F0 = F_batched(x.clone()).detach(); term1 = (F0*s).flatten(1).sum(1)
    div = torch.zeros(len(x), device=x.device)
    for _ in range(n_hutch):
        v = torch.randint(0, 2, x.shape, device=x.device).float()*2 - 1
        _, Jv = torch.autograd.functional.jvp(F_batched, x.clone(), v, create_graph=False); div += (Jv.detach()*v).flatten(1).sum(1)
    return (term1 + div/n_hutch).detach()

def net_curve2(clf, Xp, std, sigma, n=8000, nbins=50, seed=0, chunk=250):
    torch.manual_seed(seed); mid, half, sep = std(sigma); xb = Xp[torch.randperm(len(Xp), device=dev)[:n]]; xt = xb + sigma*torch.randn_like(xb)
    phi = lambda x, s: (clf(x, float(s)) - mid)/half
    with torch.no_grad(): m = phi(xt, sigma).cpu().numpy()
    mf = torch.cat([mean_force_batched(phi, score_net, xc, sigma) for xc in xt.split(chunk)]).cpu().numpy()
    edges = np.linspace(-3.5, 3.5, nbins+1); b = np.clip(np.digitize(m, edges)-1, 0, nbins-1); cnt = np.bincount(b, minlength=nbins).astype(float); est = np.zeros(nbins)
    for i in range(nbins):
        v = mf[b == i]
        if len(v) >= 20: lo, hi = np.percentile(v, [5, 95]); est[i] = np.clip(v, lo, hi).mean()          # winsorised bin mean
    k = np.exp(-0.5*(np.arange(-3, 4)/1.2)**2); est = np.convolve(est*cnt, k, 'same')/np.maximum(np.convolve(cnt, k, 'same'), 1e-9)
    return 0.5*(edges[:-1]+edges[1:]), est, cnt

print(f"{'pair':>16} {'exact sigma_bif':>16} {'net (half 1)':>13} {'net (half 2)':>13} {'split-half RMS':>15} {'vs-exact RMS':>13}")
for pair in [(2, 3), (3, 9), (5, 7), (3, 7)]:
    Xp, Yp = pair_data(pair); clf = pair_logit(pair); r = exact_census(clf, Xp, Yp); t0 = time.time()
    sg_grid = np.geomspace(max(0.3, r['sigma_bif']/3), min(30, r['sigma_bif']*3), 8)
    b1 = bisect_sigma(lambda s: zero_crossings(*net_curve2(clf, Xp, r['std'], s, seed=1)), sg_grid, iters=4)
    b2 = bisect_sigma(lambda s: zero_crossings(*net_curve2(clf, Xp, r['std'], s, seed=2)), sg_grid, iters=4)
    c1, e1, n1 = net_curve2(clf, Xp, r['std'], r['sigma_bif'], seed=3); c2, e2, n2 = net_curve2(clf, Xp, r['std'], r['sigma_bif'], seed=4)
    ex = np.interp(c1, GRID, r['ref_at'](r['sigma_bif'])[0]); ok = (np.abs(c1) < 1.5) & (n1 >= 40) & (n2 >= 40); norm = np.sqrt(np.mean(ex[ok]**2))
    print(f"{CLASSES[pair[0]]+'/'+CLASSES[pair[1]]:>16} {r['sigma_bif']:16.3f} {(b1[0] if b1 else float('nan')):13.3f} {(b2[0] if b2 else float('nan')):13.3f} "
          f"{np.sqrt(np.mean((e1[ok]-e2[ok])**2))/norm:15.3f} {np.sqrt(np.mean((0.5*(e1[ok]+e2[ok])-ex[ok])**2))/norm:13.3f}   ({time.time()-t0:.0f}s)")
    torch.cuda.empty_cache()


## What to report back
1. The two-coordinate correlation lines and scatter; the sigma_bif-by-category lines.
2. The sign-agreement count and corr(h, var difference).
3. The noise-bias ratios.
4. The fidelity table: if split-half RMS drops well below vs-exact RMS, the drift error is real; if both halves give the same sigma_bif shift, the earlier transition is real.